# ResearchLanka Kaggle Main-Branch Full Run

Import this notebook into Kaggle and run cells from top to bottom.

It will:

- clone/pull the latest `main` branch
- copy your uploaded raw dataset into the repo
- install Python + Dagster dependencies
- run the Dagster no-collection preprocessing job
- audit OpenAlex and Crossref LK authorship evidence
- build best-quality embeddings
- train Logistic Regression
- train Linear SVM
- compare model metrics
- zip outputs for download

It does **not** collect data from APIs or repositories.

## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data")
OUTPUT_ZIP = WORK_DIR / "researchlanka-kaggle-outputs.zip"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)
print("Output zip:", OUTPUT_ZIP)

## 2. Check Kaggle Dataset Exists

If this fails, your Kaggle dataset path is different. Update `DATASET_DATA_DIR` above.

In [ ]:
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 5 -type d | head -80
!test -d {DATASET_DATA_DIR} && echo "Dataset path OK" || echo "Dataset path NOT FOUND"

## 3. Clone Or Pull Latest Main Branch

In [ ]:
%cd /kaggle/working
if CODE_DIR.exists() and (CODE_DIR / ".git").exists():
    %cd /kaggle/working/code
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} code
    %cd /kaggle/working/code

!git log --oneline -3
!ls

## 4. Copy Uploaded Raw Data Into Backend

In [ ]:
%cd /kaggle/working/code/backend
!rm -rf data
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -60

## 5. Install Dependencies

This uses a Kaggle-friendly install. It avoids installing the full pinned backend requirements because those upgrade pandas/numpy and can conflict with Kaggle's preinstalled packages.

In [ ]:
%cd /kaggle/working/code/backend

!python -m pip install -r requirements.txt "protobuf<6" "google-cloud-bigquery-storage>=2.30,<3"

!python -m pip install "dagster==1.13.16" "dagster-webserver==1.13.16" "protobuf<6" "google-cloud-bigquery-storage>=2.30,<3"

!python -m pip install -e . --no-deps

!python -m pip install -e dagster-quickstart --no-deps

!python -m dagster --version

import pandas as pd
import sklearn
import pyarrow as pa
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("pyarrow", pa.__version__)


## 6. Run Dagster Pipeline Without Data Collection

This prepares existing source files and runs preprocessing through the analysis-ready dataset.

In [ ]:
%cd /kaggle/working/code/backend/dagster-quickstart

from dagster_quickstart.definitions import defs

loaded_defs = defs() if callable(defs) else defs
job = loaded_defs.get_job_def("researchlanka_no_collection_preprocessing_job")
result = job.execute_in_process()
if not result.success:
    raise RuntimeError("Dagster preprocessing job failed")


## 7. Run LK Affiliation Audits

This checks publication-time Sri Lankan institutional authorship evidence for both OpenAlex and Crossref. The audits write review queues, verified-authorship files, issue summaries, and Markdown/PDF-ready reports under `data/reports/`.

In [ ]:
%cd /kaggle/working/code/backend
!make lk-affiliation-audits PYTHON=python

from pathlib import Path
import json
import shutil
import subprocess
import pandas as pd
from IPython.display import FileLink, display

AUDITS = {
    "OpenAlex": {
        "dir": Path("data/reports/openalex_lk_affiliation_audit"),
        "summary": "lk_affiliation_audit_summary.json",
        "report": "lk_affiliation_audit_report.md",
        "verified": "verified_lk_authorships.csv",
        "review": "lk_affiliation_manual_review.csv",
        "audit_records": "lk_affiliation_audit_records.csv",
    },
    "Crossref": {
        "dir": Path("data/reports/crossref_lk_affiliation_audit"),
        "summary": "crossref_lk_affiliation_audit_summary.json",
        "report": "crossref_lk_affiliation_audit_report.md",
        "verified": "verified_lk_authorships.csv",
        "review": "crossref_lk_affiliation_manual_review.csv",
        "audit_records": "crossref_lk_affiliation_audit_records.csv",
    },
}

rows = []
for source, config in AUDITS.items():
    audit_dir = config["dir"]
    summary_path = audit_dir / config["summary"]
    report_path = audit_dir / config["report"]
    if not summary_path.exists():
        raise FileNotFoundError(f"Missing {source} audit summary: {summary_path}")

    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    overall = summary["overall"]
    impact = summary["publication_impact"]
    problems = summary["potential_problems"]

    if source == "OpenAlex":
        total_rows = overall["currently_lk_authorships"]
        issue_key = "at_least_one_issue_authorships"
        works_key = "unique_openalex_work_ids"
        candidate = overall["currently_lk_authorships"]
    else:
        total_rows = overall["audit_rows_including_authorless_works"]
        issue_key = "at_least_one_issue_authorship_rows"
        works_key = "unique_crossref_work_ids"
        candidate = overall["candidate_lk_authorships"]

    rows.extend([
        {"source": source, "metric": "Works checked", "value": f"{overall[works_key]:,}"},
        {"source": source, "metric": "Candidate LK authorships", "value": f"{candidate:,}"},
        {"source": source, "metric": "Audited author rows", "value": f"{total_rows:,}"},
        {"source": source, "metric": "Strict verified works", "value": f"{impact['strict_verified_dataset_size']:,}"},
        {"source": source, "metric": "Retained under strict rule", "value": f"{impact['percentage_retained']}%"},
        {"source": source, "metric": "Works sent to review", "value": f"{impact['records_sent_to_review']:,}"},
        {"source": source, "metric": "Authorship rows with any issue", "value": f"{problems[issue_key]['count']:,} ({problems[issue_key]['percent']}%)"},
    ])

    print(f"{source} audit files")
    for file_name in [config["report"], config["summary"], config["audit_records"], config["review"], config["verified"]]:
        file_path = audit_dir / file_name
        size_mb = file_path.stat().st_size / (1024 * 1024) if file_path.exists() else 0
        print(f"- {file_path} ({size_mb:.2f} MB)")
    print()

    pdf_path = report_path.with_suffix(".pdf")
    if shutil.which("pandoc"):
        result = subprocess.run(
            [
                "pandoc",
                str(report_path),
                "-o",
                str(pdf_path),
                "--pdf-engine=xelatex",
                "-V",
                "geometry:margin=0.75in",
                "-V",
                "fontsize=10pt",
                "-V",
                "colorlinks=true",
            ],
            text=True,
            capture_output=True,
        )
        if result.returncode == 0:
            print(f"PDF report created: {pdf_path}")
        else:
            print(f"PDF report skipped for {source}; pandoc failed:")
            print(result.stderr[-1200:])
    else:
        print(f"PDF report skipped for {source}; pandoc is not installed in this Kaggle image.")
    print()

metrics = pd.DataFrame(rows)
display(metrics)

for source, config in AUDITS.items():
    audit_dir = config["dir"]
    display(FileLink(str(audit_dir / config["summary"])))
    display(FileLink(str(audit_dir / config["report"])))
    pdf_path = (audit_dir / config["report"]).with_suffix(".pdf")
    if pdf_path.exists():
        display(FileLink(str(pdf_path)))


## 7. Verify Preprocessing Outputs

In [ ]:
%cd /kaggle/working/code/backend
!ls -lh data/processed/repositories_combined.csv
!ls -lh data/processed/sljol.csv
!ls -lh data/processed/common/common_publications_final.csv
!ls -lh data/processed/common/common_publications_final_2016_2026_analysis_ready.csv

import pandas as pd
paths = [
    'data/processed/common/common_publications_final.csv',
    'data/processed/common/common_publications_final_2016_2026_analysis_ready.csv',
]
for path in paths:
    frame = pd.read_csv(path, nrows=5)
    total = sum(1 for _ in open(path, encoding='utf-8')) - 1
    print(path, 'rows=', total, 'columns=', len(frame.columns))

## 8. Build Best-Quality Embeddings

Full text fields, trigrams, larger vocabulary, 512 dimensions, no row limit.

In [ ]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python \
  EMBED_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  EMBED_MAX_FEATURES=100000 \
  EMBED_NGRAM_MAX=3 \
  EMBED_DIM=512
!ls -lh data/models/publication_text_embeddings.parquet data/models/publication_text_embedding_model.joblib data/models/publication_text_embeddings_summary.txt

## 9. Train Best-Quality Logistic Regression

In [ ]:
%cd /kaggle/working/code/backend
!make train-logreg PYTHON=python \
  LOGREG_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  LOGREG_MAX_FEATURES=100000 \
  LOGREG_NGRAM_MAX=3 \
  LOGREG_MAX_ITER=2000
!cat data/models/logistic_regression_primary_domain_metrics.txt

## 10. Train Best-Quality Linear SVM

If Kaggle RAM fails, rerun this cell with `--max-features 50000`.

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --ngram-max 3 \
  --max-features 100000 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --max-iter 5000 \
  --test-size 0.2
!cat data/models/linear_svm_primary_domain_metrics.txt

## 12. Verify Modeling And Audit Outputs

In [ ]:
%cd /kaggle/working/code/backend
from pathlib import Path

required = [
    "data/models/publication_text_embeddings.parquet",
    "data/models/publication_text_embedding_model.joblib",
    "data/models/logistic_regression_primary_domain.joblib",
    "data/models/logistic_regression_primary_domain_metrics.txt",
    "data/models/linear_svm_primary_domain.joblib",
    "data/models/linear_svm_primary_domain_metrics.txt",
    "data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_summary.json",
    "data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_report.md",
    "data/reports/crossref_lk_affiliation_audit/crossref_lk_affiliation_audit_summary.json",
    "data/reports/crossref_lk_affiliation_audit/crossref_lk_affiliation_audit_report.md",
]

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing required outputs:\n" + "\n".join(missing))

for p in required:
    path = Path(p)
    print(f"OK {p} ({path.stat().st_size / (1024*1024):.2f} MB)")


## 12. Zip Outputs For Download

In [ ]:
%cd /kaggle/working/code/backend
!rm -f /kaggle/working/researchlanka-kaggle-outputs.zip
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models data/reports
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip

Download this file from the Kaggle output panel:

```text
/kaggle/working/researchlanka-kaggle-outputs.zip
```